In [2]:
import os
import time
import json
import numpy as np
import pandas as pd
import pm4py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import optuna
import warnings

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
torch.manual_seed(1)
np.random.seed(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final

# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, True)
    test_df = build_prefix_df(test_split, prefix_lengths, True)

    return train_df, test_df



In [7]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
#GLOBAL_prefix_lengths = [100, 150, 200, 300, 400, 500, 600, 700, 800]

train_df_2013, test_df_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 3289.95it/s]


In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

def encode_and_format(df, activity_to_idx, n_classes, prefix_length):
    # 1. Convert subtrace strings to integers
    # Shape: (num_samples, prefix_length)
    X_int = np.array([[activity_to_idx[act] for act in sub] for sub in df['subtrace']])
    
    # 2. Manual One-Hot Encoding for the sequence
    # Shape: (num_samples, prefix_length, n_classes)
    X_ohe = np.zeros((len(X_int), prefix_length, n_classes), dtype=np.float32)
    for i, seq in enumerate(X_int):
        for t, act_idx in enumerate(seq):
            X_ohe[i, t, act_idx] = 1.0
            
    # 3. Encode Target (Next Activity)
    y = np.array([activity_to_idx[act] for act in df['next_activity']])
    
    return torch.tensor(X_ohe), torch.tensor(y, dtype=torch.long)

class ResourceLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ResourceLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # out shape: (batch, seq_len, hidden_dim)
        _, (hn, _) = self.lstm(x) 
        # Use the last hidden state to predict
        return self.fc(hn[-1])

# 1. Prepare global vocabulary from the original dataframe
all_activities = sorted(df_2013['concept:name'].unique())
act_to_idx = {act: i for i, act in enumerate(all_activities)}
n_classes = len(all_activities)

results = {}

for length in GLOBAL_prefix_lengths:
    print(f"\n--- Testing Prefix Length: {length} ---")
    
    # Use your existing pre-processing logic
    train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    test_df = test_df_2013[test_df_2013['prefix_length'] == length]
    
    if train_df.empty or test_df.empty:
        continue

    # Encode
    X_train, y_train = encode_and_format(train_df, act_to_idx, n_classes, length)
    X_test, y_test = encode_and_format(test_df, act_to_idx, n_classes, length)
    
    # Data Loaders
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    
    # Initialize Model
    model = ResourceLSTM(input_dim=n_classes, hidden_dim=64, output_dim=n_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # Quick Training Loop (e.g., 10 epochs)
    model.train()
    for epoch in range(10):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        predictions = torch.argmax(test_outputs, dim=1)
        accuracy = (predictions == y_test).float().mean().item()
        results[length] = accuracy
        print(f"Accuracy for length {length}: {accuracy:.4f}")

print("\nFinal Results Table:", results)


--- Testing Prefix Length: 10 ---
Accuracy for length 10: 0.6961

--- Testing Prefix Length: 20 ---
Accuracy for length 20: 0.6954

--- Testing Prefix Length: 30 ---
Accuracy for length 30: 0.7011

--- Testing Prefix Length: 40 ---
Accuracy for length 40: 0.7029

--- Testing Prefix Length: 50 ---
Accuracy for length 50: 0.7038

--- Testing Prefix Length: 75 ---
Accuracy for length 75: 0.6922

--- Testing Prefix Length: 100 ---
Accuracy for length 100: 0.6952

--- Testing Prefix Length: 125 ---
Accuracy for length 125: 0.6913

--- Testing Prefix Length: 150 ---
Accuracy for length 150: 0.6887

Final Results Table: {10: 0.6960506439208984, 20: 0.6954004764556885, 30: 0.7011246681213379, 40: 0.7028927803039551, 50: 0.7037793397903442, 75: 0.6922103762626648, 100: 0.6952444911003113, 125: 0.6913372874259949, 150: 0.688746452331543}


In [ ]:
# Training and evaluating model
def train_and_evaluate(model, train_loader, X_test, y_test, device, epochs=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation        
    model.eval()
    with torch.no_grad():
        X_test, y_test = X_test.to(device), y_test.to(device)
        preds = torch.argmax(model(X_test), dim=1)
        accuracy = (preds == y_test).float().mean().item()
        
    return accuracy